# Telemetry XGBoost End-to-End

This notebook trains the telemetry branch for the industrial demo:

```text
FD001 sensor history
  -> validation-safe feature engineering
  -> XGBoost RUL regression
  -> XGBoost failure-risk classification
  -> telemetry contract + saved artifacts
```

The notebook keeps engine-level separation, so validation engines never leak
into training.


## 1. Environment


In [6]:
# Install only if the server is missing packages.
%pip install -U numpy pandas scikit-learn xgboost matplotlib seaborn joblib



[notice] A new release of pip is available: 25.2 -> 26.1.2
[notice] To update, run: python3.12 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [7]:
from pathlib import Path
import json
import random

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import GroupShuffleSplit
from xgboost import XGBClassifier, XGBRegressor

SEED = 42
RUL_CAP = 125
CRITICAL_RUL = 15
WARNING_RUL = 30
ROLLING_WINDOWS = (5, 10, 20)

random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid")

DATA_CANDIDATES = (
    Path("data/CMAPSSData"),
    Path("data"),
    Path("../data/CMAPSSData"),
    Path("../data"),
    Path("CMAPSSData"),
)

ARTIFACT_DIR = Path("artifacts/telemetry_xgboost")
FIGURE_DIR = ARTIFACT_DIR / "figures"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print("Working directory:", Path.cwd().resolve())
print("Artifacts:", ARTIFACT_DIR.resolve())


Working directory: /workspace/notebooks
Artifacts: /workspace/notebooks/artifacts/telemetry_xgboost


## 2. Load FD001


In [8]:
COLUMNS = (
    ["unit", "cycle"]
    + [f"op_setting_{index}" for index in range(1, 4)]
    + [f"sensor_{index}" for index in range(1, 22)]
)


def resolve_data_dir() -> Path:
    required = ("train_FD001.txt", "test_FD001.txt", "RUL_FD001.txt")
    for candidate in DATA_CANDIDATES:
        if all((candidate / name).is_file() for name in required):
            return candidate
    searched = "\n".join(str(path.resolve()) for path in DATA_CANDIDATES)
    raise FileNotFoundError(f"FD001 files not found. Searched:\n{searched}")


DATA_DIR = resolve_data_dir()
train_raw = pd.read_csv(
    DATA_DIR / "train_FD001.txt",
    sep=r"\s+",
    header=None,
    names=COLUMNS,
)
test_raw = pd.read_csv(
    DATA_DIR / "test_FD001.txt",
    sep=r"\s+",
    header=None,
    names=COLUMNS,
)
test_rul_raw = pd.read_csv(
    DATA_DIR / "RUL_FD001.txt",
    sep=r"\s+",
    header=None,
    names=["RUL"],
)

assert train_raw.shape[1] == 26
assert test_raw.shape[1] == 26
assert test_raw["unit"].nunique() == len(test_rul_raw)
print("Data directory:", DATA_DIR.resolve())
print("Train rows:", len(train_raw))
print("Test rows:", len(test_raw))
print("Train engines:", train_raw["unit"].nunique())
print("Test engines:", test_raw["unit"].nunique())

Data directory: /workspace/notebooks/data/CMAPSSData
Train rows: 20631
Test rows: 13096
Train engines: 100
Test engines: 100


## 3. Feature Engineering


In [9]:
train_labeled = train_raw.copy()
max_cycle = train_labeled.groupby("unit")["cycle"].transform("max")
train_labeled["RUL_raw"] = max_cycle - train_labeled["cycle"]
train_labeled["RUL"] = train_labeled["RUL_raw"].clip(upper=RUL_CAP)
train_labeled["failure_within_horizon"] = (
    train_labeled["RUL_raw"] <= WARNING_RUL
).astype(int)

candidate_features = [column for column in COLUMNS if column != "unit"]
feature_std = train_raw[candidate_features].std()
constant_columns = feature_std[feature_std <= 1e-8].index.tolist()
sensor_features = [
    column
    for column in candidate_features
    if column.startswith("sensor_") and column not in constant_columns
]

print("Constant columns:", constant_columns)
print("Sensor features:", len(sensor_features))


Constant columns: ['op_setting_3', 'sensor_1', 'sensor_5', 'sensor_10', 'sensor_16', 'sensor_18', 'sensor_19']
Sensor features: 15


In [10]:
def engineer_features(frame: pd.DataFrame) -> pd.DataFrame:
    result = frame.sort_values(["unit", "cycle"]).copy()
    grouped = result.groupby("unit", sort=False)
    derived = {}

    for sensor in sensor_features:
        derived[f"{sensor}_delta"] = grouped[sensor].diff().fillna(0.0)

    for window in ROLLING_WINDOWS:
        for sensor in sensor_features:
            rolling = grouped[sensor].rolling(window=window, min_periods=1)
            derived[f"{sensor}_mean_{window}"] = (
                rolling.mean().reset_index(level=0, drop=True)
            )
            derived[f"{sensor}_std_{window}"] = (
                rolling.std().reset_index(level=0, drop=True).fillna(0.0)
            )

    return pd.concat([result, pd.DataFrame(derived, index=result.index)], axis=1)


train_features = engineer_features(train_labeled)
test_features = engineer_features(test_raw)

excluded = {"unit", "RUL", "RUL_raw", "failure_within_horizon"}
feature_columns = [
    column for column in train_features.columns if column not in excluded
]

assert not train_features[feature_columns].isna().any().any()
print("Feature count:", len(feature_columns))


Feature count: 130


## 4. Train / Validation Split


In [11]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_index, validation_index = next(
    splitter.split(train_features, groups=train_features["unit"])
)

development = train_features.iloc[train_index].copy()
validation = train_features.iloc[validation_index].copy()

X_train = development[feature_columns]
y_train_rul = development["RUL"]
y_train_risk = development["failure_within_horizon"]
X_validation = validation[feature_columns]
y_validation_rul = validation["RUL"]
y_validation_risk = validation["failure_within_horizon"]

scale_pos_weight = (
    float((y_train_risk == 0).sum()) / max(float((y_train_risk == 1).sum()), 1.0)
)

assert set(development["unit"]).isdisjoint(set(validation["unit"]))
print("Training engines:", development["unit"].nunique())
print("Validation engines:", validation["unit"].nunique())
print("scale_pos_weight:", round(scale_pos_weight, 3))


Training engines: 80
Validation engines: 20
scale_pos_weight: 5.678


## 5. Train XGBoost Models


In [15]:
rul_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist",
)
risk_model = XGBClassifier(
    objective="binary:logistic",
    eval_metric="logloss",
    n_estimators=450,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    random_state=SEED,
    n_jobs=-1,
    tree_method="hist",
)

rul_model.fit(X_train, y_train_rul)
risk_model.fit(X_train, y_train_risk)

validation_rul_pred = rul_model.predict(X_validation)
validation_risk_prob = risk_model.predict_proba(X_validation)[:, 1]
validation_risk_pred = (validation_risk_prob >= 0.5).astype(int)

rmse = mean_squared_error(y_validation_rul, validation_rul_pred) ** 0.5
mae = mean_absolute_error(y_validation_rul, validation_rul_pred)
r2 = r2_score(y_validation_rul, validation_rul_pred)
auc = roc_auc_score(y_validation_risk, validation_risk_prob)

print({"rmse": rmse, "mae": mae, "r2": r2, "auc": auc})
print(classification_report(y_validation_risk, validation_risk_pred, digits=3))

{'rmse': 14.35794993438685, 'mae': 9.764127731323242, 'r2': 0.8815359473228455, 'auc': 0.9933969144460028}
              precision    recall  f1-score   support

           0      0.985     0.977     0.981      3450
           1      0.875     0.918     0.896       620

    accuracy                          0.968      4070
   macro avg      0.930     0.947     0.938      4070
weighted avg      0.968     0.968     0.968      4070



## 6. Official Test Prediction and Artifacts


In [17]:
test_labeled = test_features.copy()

# RUL_FD001.txt is one value per engine, in the same order as sorted test units
test_units = sorted(test_raw["unit"].unique())
rul_lookup = pd.Series(test_rul_raw["RUL"].values, index=test_units)

test_labeled["RUL"] = test_labeled["unit"].map(rul_lookup)

if test_labeled["RUL"].isna().any():
    raise ValueError("Some test units did not receive an RUL value.")

test_labeled["failure_within_horizon"] = (
    test_labeled["RUL"] <= WARNING_RUL
).astype(int)

test_rul_pred = rul_model.predict(test_labeled[feature_columns])
test_risk_prob = risk_model.predict_proba(test_labeled[feature_columns])[:, 1]
test_risk_pred = (test_risk_prob >= 0.5).astype(int)

test_results = pd.DataFrame(
    {
        "unit": test_labeled["unit"],
        "cycle": test_labeled["cycle"],
        "true_rul": test_labeled["RUL"],
        "predicted_rul": test_rul_pred,
        "true_risk": test_labeled["failure_within_horizon"],
        "predicted_risk": test_risk_pred,
        "risk_probability": test_risk_prob,
    }
)

test_results.to_csv(ARTIFACT_DIR / "test_results.csv", index=False)
joblib.dump(rul_model, ARTIFACT_DIR / "xgb_rul_model.joblib")
joblib.dump(risk_model, ARTIFACT_DIR / "xgb_risk_model.joblib")

(ARTIFACT_DIR / "feature_columns.json").write_text(
    json.dumps(feature_columns, indent=2),
    encoding="utf-8",
)

telemetry_contract = {
    "scenario_id": "FD001-TELEMETRY-001",
    "asset_id": "CMAPSS_FD001",
    "outputs": {
        "rul": float(test_results["predicted_rul"].iloc[0]),
        "failure_risk": float(test_results["risk_probability"].iloc[0]),
        "risk_label": int(test_results["predicted_risk"].iloc[0]),
    },
    "evidence": {
        "model_rul": "xgb_rul_model.joblib",
        "model_risk": "xgb_risk_model.joblib",
        "feature_count": len(feature_columns),
        "validation_auc": float(auc),
    },
}

(ARTIFACT_DIR / "telemetry_contract_example.json").write_text(
    json.dumps(telemetry_contract, indent=2),
    encoding="utf-8",
)

print("Saved artifacts to:", ARTIFACT_DIR.resolve())
print(json.dumps(telemetry_contract, indent=2))

Saved artifacts to: /workspace/notebooks/artifacts/telemetry_xgboost
{
  "scenario_id": "FD001-TELEMETRY-001",
  "asset_id": "CMAPSS_FD001",
  "outputs": {
    "rul": 124.41255950927734,
    "failure_risk": 1.4388966519618407e-05,
    "risk_label": 0
  },
  "evidence": {
    "model_rul": "xgb_rul_model.joblib",
    "model_risk": "xgb_risk_model.joblib",
    "feature_count": 130,
    "validation_auc": 0.9933969144460028
  }
}
